In [1]:
import numpy as np
import h5py
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import DataLoader
import torch
import os

In [2]:
from jdml.dataset._patch_dataset import ScaleRotateCropPatchDataset
from jdml.models._flexible_CNN_v4 import CNNModel, get_optimizer, get_loss, train_model
from stemplot import imshow

In [3]:
def load_h5(filename):
    current_folder = os.getcwd()
    parent = os.path.dirname(current_folder)
    data_path = parent+f"\\data\\training datasets\\{filename}"
    
    with h5py.File(data_path, 'r') as f:
        train_data = f['train/data'][:]
        train_lbs = f['train/labels'][:]
        val_data = f['val/data'][:]
        val_lbs = f['val/labels'][:]
        test_data = f['test/data'][:]
        test_lbs = f['test/labels'][:]
    return train_data, train_lbs, val_data, val_lbs, test_data, test_lbs

In [4]:
train_data, train_lbs, val_data, val_lbs, test_data, test_lbs = load_h5('square_system_dataset.h5')

In [5]:
train_lbs -= train_lbs.min()
val_lbs -= val_lbs.min()
test_lbs -= test_lbs.min()

In [7]:
train_dataset = ScaleRotateCropPatchDataset(train_data, train_lbs)
val_dataset = ScaleRotateCropPatchDataset(val_data, val_lbs)

In [8]:
train_loader = DataLoader(train_dataset, batch_size = 8, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 8, shuffle = False)

In [9]:
model = CNNModel(
    input_dim=(8, 64, 64),  
    num_classes=3,          
    conv_layers=[(32, 3, 1, 1, True), (64, 3, 1, 1, False), (128, 3, 1, 1, True)],
    fc_layers=[512, 256],
    pool_config=('max', 2, 2),
    dropout=0.2
)
model.summary()

CNN Model Architecture
Input: (8, 64, 64)
Output Classes: 3
--------------------------------------------------------------------------------
Convolutional Layers:
  Conv1: 8 -> 32, kernel=3, stride=1, padding=1
          Output: (32, 64, 64)
  Pool1: max, kernel=2, stride=2
          Output: (32, 32, 32)
  Conv2: 32 -> 64, kernel=3, stride=1, padding=1
          Output: (64, 32, 32)
  Conv3: 64 -> 128, kernel=3, stride=1, padding=1
          Output: (128, 32, 32)
  Pool3: max, kernel=2, stride=2
          Output: (128, 16, 16)
--------------------------------------------------------------------------------
Flatten: (128, 16, 16) -> 32768
--------------------------------------------------------------------------------
Fully Connected Layers:
  FC1: 32768 -> 512 (dropout=0.2)
  FC2: 512 -> 256 (dropout=0.2)
  Classifier: 256 -> 3
--------------------------------------------------------------------------------
Total Parameters: 17,004,963
Trainable Parameters: 17,004,963


In [10]:
optimizer = get_optimizer(model, 'adam', lr=0.001, weight_decay=1e-4)
criterion = get_loss('crossentropy')
scheduler = StepLR(optimizer, step_size=10, gamma=0.5)

In [11]:
history = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=5,
    optimizer=optimizer,
    criterion=criterion,
    scheduler=scheduler,
    device='cuda'
)


Epoch 1/5
------------------------------------------------------------


Train Loss: 2.1535 | Train Acc: 61.46%
Val Loss:   0.2426 | Val Acc:   91.67%
Learning Rate: 0.001000
✓ New best model saved! (Val Loss: 0.2426)

Epoch 2/5
------------------------------------------------------------


Train Loss: 0.4831 | Train Acc: 85.42%
Val Loss:   0.1620 | Val Acc:   96.67%
Learning Rate: 0.001000
✓ New best model saved! (Val Loss: 0.1620)

Epoch 3/5
------------------------------------------------------------


Train Loss: 0.2390 | Train Acc: 93.96%
Val Loss:   0.0110 | Val Acc:   100.00%
Learning Rate: 0.001000
✓ New best model saved! (Val Loss: 0.0110)

Epoch 4/5
------------------------------------------------------------


Train Loss: 0.1527 | Train Acc: 94.79%
Val Loss:   0.2435 | Val Acc:   90.00%
Learning Rate: 0.001000

Epoch 5/5
------------------------------------------------------------


Train Loss: 0.2592 | Train Acc: 93.12%
Val Loss:   0.1831 | Val Acc:   91.67%
Learning Rate: 0.001000

Training complete!
Best validation loss: 0.0110
